In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

from create_engine import ENGINE


# 1. LOAD DATA

query = "SELECT * FROM IDS_DATA_AFTER_PREPROCESSING"
df = pd.read_sql(query, ENGINE)

print("Dataset Shape:", df.shape)


# 2. FEATURES & TARGET

X = df.drop("label", axis=1)
y = df["label"]


# 3. TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=101,
    stratify=y
)

print("\nTraining Distribution Before SMOTE:")
print(y_train.value_counts().sort_index())


# 4. TARGETED SMOTE

smote = SMOTE(
    sampling_strategy={
        9 : 400,
        8 : 500,
        13 : 500
    },
    random_state=42,
    k_neighbors=5
)

X_train, y_train = smote.fit_resample(
    X_train,
    y_train
)

print("\nTraining Distribution After SMOTE:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nTraining Shape:", X_train.shape)


# 5. SAMPLE FOR TUNING

sample_size = min(150000, len(X_train))

X_train_sample = X_train.sample(
    n=sample_size,
    random_state=42
)

y_train_sample = y_train.loc[X_train_sample.index]

print("\nTuning Sample Shape:", X_train_sample.shape)
# 6. XGBOOST MODEL

xgb = XGBClassifier(
    objective="multi:softmax",
    num_class=y.nunique(),
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42
    
)

pipe = Pipeline([
    ("model", xgb)
])


# 7. PARAM GRID

param_dist = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [4, 6, 8],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.7, 0.8, 1.0],
    "model__colsample_bytree": [0.5, 0.7, 1.0]

}


# 8. RANDOM SEARCH CV

search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=15,
    cv=5,
    scoring="recall_weighted",
    refit=True,
    verbose=2,
    n_jobs=-1,
    random_state=42
)
# 9. HYPERPARAMETER TUNING

search.fit(
    X_train_sample,
    y_train_sample
)

print("\nBest Parameters:")
print(search.best_params_)

print("\nBest CV Recall:")
print(search.best_score_)


# 10. FINAL MODEL

best_model = search.best_estimator_

best_model.fit(
    X_train,
    y_train
)
# 11. PREDICTIONS

y_pred = best_model.predict(X_test)


# 12. EVALUATION

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

2026-08-22 23:38:32,647 - INFO - Connection Built Successfully


Connecting to DB: localhost/cti_fyp_2
Dataset Shape: (1035656, 78)

Training Distribution Before SMOTE:
label
0     414975
1       1546
2      66586
3       5353
4     111333
5       3576
6       3995
7       2789
8          8
9         34
10    111316
11      2092
12       955
13         8
14       393
Name: count, dtype: int64

Training Distribution After SMOTE:
label
0     414975
1       1546
2      66586
3       5353
4     111333
5       3576
6       3995
7       2789
8        500
9        400
10    111316
11      2092
12       955
13       500
14       393
Name: count, dtype: int64

Training Shape: (726309, 77)

Tuning Sample Shape: (150000, 77)
Fitting 5 folds for each of 15 candidates, totalling 75 fits

Best Parameters:
{'model__subsample': 1.0, 'model__n_estimators': 200, 'model__max_depth': 4, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.7}

Best CV Recall:
0.9980466666666666

Confusion Matrix:
[[177622    195      0      0      1     16      2      0      0    

In [2]:

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))



Confusion Matrix:
[[177622    195      0      0      1     16      2      0      0      0
      10      0      1      0      0]
 [    43    619      0      0      0      0      0      0      0      0
       0      0      0      0      0]
 [     0      0  28537      0      0      0      0      0      0      0
       0      0      0      0      0]
 [     1      0      0   2284      4      4      0      0      0      0
       0      0      1      0      0]
 [     2      0      0      8  47701      2      0      0      0      0
       0      0      2      0      0]
 [     4      0      0      1      0   1524      3      0      0      0
       0      0      1      0      0]
 [     3      0      0      0      0      1   1708      0      0      0
       0      0      0      0      0]
 [     0      0      0      0      0      0      0   1195      0      0
       0      0      0      0      0]
 [     0      0      0      0      0      0      0      0      3      0
       0      0      0      0

In [3]:

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    177847
           1       0.76      0.94      0.84       662
           2       1.00      1.00      1.00     28537
           3       1.00      1.00      1.00      2294
           4       1.00      1.00      1.00     47715
           5       0.99      0.99      0.99      1533
           6       1.00      1.00      1.00      1712
           7       1.00      1.00      1.00      1195
           8       1.00      1.00      1.00         3
           9       1.00      0.71      0.83        14
          10       1.00      1.00      1.00     47707
          11       1.00      1.00      1.00       896
          12       0.71      0.98      0.82       410
          13       1.00      0.25      0.40         4
          14       0.44      0.05      0.09       168

    accuracy                           1.00    310697
   macro avg       0.93      0.86      0.86    310697
we

In [4]:
import joblib

In [6]:
joblib.dump(best_model, 'XGBoost_model(smote)_3rd_milestone.joblib')

['XGBoost_model(smote)_3rd_milestone.joblib']

In [8]:
import numpy as np
import pandas as pd
import joblib

In [9]:
model = joblib.load('XGBoost_model(smote)_3rd_milestone.joblib')

In [10]:
for label in df['label'].unique():
    indices = df[df['label'] == label].index.tolist()
    print(f"\n{label} ({len(indices)} rows):")
    print(f"  First 5 indices: {indices[:5]}")
    print(f"  Last 5 indices:  {indices[-5:]}")


0 (592822 rows):
  First 5 indices: [0, 1, 2, 3, 4]
  Last 5 indices:  [592817, 592818, 592819, 592820, 592821]

4 (159048 rows):
  First 5 indices: [592822, 592823, 592824, 592828, 592829]
  Last 5 indices:  [1035636, 1035642, 1035643, 1035649, 1035651]

2 (95123 rows):
  First 5 indices: [592825, 592826, 592832, 592835, 592842]
  Last 5 indices:  [1035646, 1035647, 1035648, 1035653, 1035654]

11 (2988 rows):
  First 5 indices: [592827, 592840, 592914, 592929, 593436]
  Last 5 indices:  [1035065, 1035181, 1035266, 1035544, 1035561]

10 (159023 rows):
  First 5 indices: [592831, 592833, 592836, 592837, 592838]
  Last 5 indices:  [1035637, 1035640, 1035644, 1035650, 1035652]

3 (7647 rows):
  First 5 indices: [592834, 592870, 592942, 592962, 593023]
  Last 5 indices:  [1035443, 1035473, 1035474, 1035509, 1035638]

5 (5109 rows):
  First 5 indices: [592887, 593119, 593311, 593377, 593592]
  Last 5 indices:  [1034999, 1035113, 1035210, 1035262, 1035599]

7 (3984 rows):
  First 5 indices:

In [18]:
# Define the specific attacks you want to check
target_labels = [11]

for label in target_labels:
    # Safely check if the label actually exists in your data
    if label in df['label'].unique():
        indices = df[df['label'] == label].index.tolist()
        print(f"\nAttack Label {label} ({len(indices)} rows):")
        print(f"  First 5 indices: {indices[:5]}")
        print(f"  Last 5 indices: {indices[-5:]}")
    else:
        print(f"\nAttack Label {label} not found in dataset.")



Attack Label 11 (2988 rows):
  First 5 indices: [592827, 592840, 592914, 592929, 593436]
  Last 5 indices: [1035065, 1035181, 1035266, 1035544, 1035561]


In [12]:
encoder = joblib.load('preprocessed_encoder.joblib')

In [13]:
print(dict(enumerate(encoder.classes_))) 

{0: 'BENIGN', 1: 'Bot', 2: 'DDoS', 3: 'DoS GoldenEye', 4: 'DoS Hulk', 5: 'DoS Slowhttptest', 6: 'DoS slowloris', 7: 'FTP-Patator', 8: 'Heartbleed', 9: 'Infiltration', 10: 'PortScan', 11: 'SSH-Patator', 12: 'Web Attack - Brute Force', 13: 'Web Attack - Sql Injection', 14: 'Web Attack - XSS'}


In [20]:
import time

sample = df.iloc[991524]
X_sample = sample.drop("label").values.reshape(1, -1)

start = time.perf_counter()
pred = model.predict(X_sample)
end = time.perf_counter()

print("Predicted:", encoder.inverse_transform(pred)[0])
print("Actual:   ", encoder.inverse_transform([int(sample['label'])])[0])
print(f"Prediction time: {(end - start)*1000:.4f} ms")

Predicted: BENIGN
Actual:    Web Attack - Sql Injection
Prediction time: 9.1192 ms


In [6]:
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

from create_engine import ENGINE

2026-08-08 14:35:18,693 - INFO - Connection Built Successfully


Connecting to DB: localhost/cti_fyp_2


In [7]:
query = "SELECT * FROM IDS_DATA_AFTER_PREPROCESSING"
df = pd.read_sql(query, ENGINE)

In [22]:
import time

df_sample = df.sample(n=1000, random_state=101)
X_sample = df_sample.drop('label', axis=1)
y_actual = encoder.inverse_transform(df_sample['label'].astype(int).values)  # decode labels upfront
actual_indices = df_sample.index.tolist()  # get actual dataset indices

times = []
correct = 0

print(f"{'Index':<8} {'Actual':<25} {'Predicted':<25} {'Time (ms)':<12} {'Result'}")
print("-" * 80)

for i in range(len(X_sample)):
    X_row = X_sample.iloc[[i]]

    start = time.perf_counter()
    pred = model.predict(X_row)
    end = time.perf_counter()

    elapsed_ms = (end - start) * 1000
    times.append(elapsed_ms)

    predicted_label = encoder.inverse_transform(pred)[0]
    actual_label = y_actual[i]
    status = '✅' if predicted_label == actual_label else '❌'
    if predicted_label == actual_label:
        correct += 1

    print(f"{actual_indices[i]:<8} {actual_label:<25} {predicted_label:<25} {elapsed_ms:<12.4f} {status}", flush=True)

    wait = max(elapsed_ms / 1000, 0.3)
    time.sleep(wait)

print("-" * 80)
print(f"\nAverage prediction time: {sum(times)/len(times):.4f} ms")
print(f"Total rows:              {len(times)}")
print(f"Correct predictions:     {correct}/{len(times)}")

Index    Actual                    Predicted                 Time (ms)    Result
--------------------------------------------------------------------------------
950222   DoS Hulk                  DoS Hulk                  12.7692      ✅
858074   PortScan                  PortScan                  33.0336      ✅
256323   BENIGN                    BENIGN                    23.1590      ✅
427893   BENIGN                    BENIGN                    28.6151      ✅
703452   DoS Hulk                  DoS Hulk                  12.6267      ✅
875548   PortScan                  PortScan                  22.5485      ✅
843466   PortScan                  PortScan                  13.8606      ✅
806121   DoS Hulk                  DoS Hulk                  10.2784      ✅
353651   BENIGN                    BENIGN                    16.1242      ✅
717887   PortScan                  PortScan                  25.5454      ✅
533558   BENIGN                    BENIGN                    38.9089      ✅
95